# Coffee Shop Dynamic Perception — Colab Demo

This notebook runs the static-ceiling-camera dynamic perception pipeline on a coffee-shop video.

**What it does:**
- Detects and tracks people with YOLOv8n + ByteTrack
- Computes optical flow with RAFT-Small every 5 frames
- Produces an annotated output video + a JSONL scene log

**Runtime:** ~2-4 fps on a T4 GPU for 720p video. Enable GPU in *Runtime -> Change runtime type -> T4 GPU*.

In [ ]:
# Cell 1 -- Install dependencies
# opencv-python-headless is used in Colab (no display server)
!pip install -q ultralytics torchvision opencv-python-headless tqdm

In [ ]:
# Cell 2 -- Mount Google Drive OR upload a file directly
import os

USE_DRIVE = True  # Set to False to upload a file instead

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    # Edit this path to point to your video on Drive:
    INPUT_VIDEO = '/content/drive/MyDrive/coffee_shop.mp4'
    OUTPUT_VIDEO = '/content/drive/MyDrive/coffee_shop_annotated.mp4'
    LOG_PATH = '/content/drive/MyDrive/coffee_shop_scene_log.jsonl'
else:
    from google.colab import files
    print('Upload your video file:')
    uploaded = files.upload()
    INPUT_VIDEO = list(uploaded.keys())[0]
    OUTPUT_VIDEO = 'coffee_shop_annotated.mp4'
    LOG_PATH = 'coffee_shop_scene_log.jsonl'

print(f'Input  : {INPUT_VIDEO}')
print(f'Output : {OUTPUT_VIDEO}')
print(f'Log    : {LOG_PATH}')

In [ ]:
# Cell 3 -- Clone the repo (or copy pipeline.py to /content)
# If running from the repo directly, skip this cell.
import sys
import os

REPO_URL = 'https://github.com/YOUR_ORG/wheelchair-research.git'  # update before use
PIPELINE_PATH = '/content/wheelchair-research/demo/coffee-shop-demo'

if not os.path.exists(PIPELINE_PATH):
    os.system(f'git clone --depth 1 {REPO_URL} /content/wheelchair-research')

sys.path.insert(0, PIPELINE_PATH)
print('Pipeline path added to sys.path:', PIPELINE_PATH)

In [ ]:
# Cell 4 -- Instantiate pipeline and process the video
import json
import time
import cv2
import numpy as np
from tqdm import tqdm
from pipeline import DynamicPerceptionPipeline, DEFAULT_CONFIG

config = {
    **DEFAULT_CONFIG,
    'device': 'auto',
    'flow_interval': 5,
    'theta': 2.0,
    'theta_stop': 8.0,
}

print('Initialising pipeline...')
pipeline = DynamicPerceptionPipeline(config)

cap = cv2.VideoCapture(INPUT_VIDEO)
if not cap.isOpened():
    raise FileNotFoundError(f'Cannot open video: {INPUT_VIDEO}')

src_fps  = cap.get(cv2.CAP_PROP_FPS) or 30.0
n_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
frame_w  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
frame_h  = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
print(f'Video: {frame_w}x{frame_h} @ {src_fps:.1f} fps, {n_frames} frames')

fourcc = cv2.VideoWriter_fourcc(*'mp4v')
writer = cv2.VideoWriter(OUTPUT_VIDEO, fourcc, src_fps, (frame_w, frame_h))

# Keep first 5 annotated frames for inline display
preview_frames = []
scene_log = []

t0 = time.perf_counter()
for idx in tqdm(range(n_frames), unit='frame'):
    ret, frame = cap.read()
    if not ret:
        break
    elapsed = time.perf_counter() - t0
    fps_now = idx / elapsed if elapsed > 0 else 0.0

    result = pipeline.process_frame(frame, idx, fps=fps_now)
    writer.write(result.annotated_frame)
    scene_log.append(result.scene_state)

    if idx < 5:
        preview_frames.append(result.annotated_frame.copy())

cap.release()
writer.release()

with open(LOG_PATH, 'w') as f:
    for entry in scene_log:
        f.write(json.dumps(entry) + '\n')

total = time.perf_counter() - t0
print(f'Done: {idx+1} frames in {total:.1f}s ({(idx+1)/total:.1f} fps avg)')
print(f'Output video: {OUTPUT_VIDEO}')
print(f'Scene log:    {LOG_PATH}')

In [ ]:
# Cell 5 -- Display the first 5 annotated frames inline
from IPython.display import display, Image as IPImage
import cv2

print(f'Showing {len(preview_frames)} preview frames:')
for i, bgr in enumerate(preview_frames):
    _, buf = cv2.imencode('.png', bgr)
    print(f'--- Frame {i} ---')
    display(IPImage(data=buf.tobytes()))

In [ ]:
# Cell 6 -- Load and display the first 10 lines of the JSONL log as a table
import json
import pandas as pd
from IPython.display import display

rows = []
with open(LOG_PATH) as f:
    for i, line in enumerate(f):
        if i >= 10:
            break
        entry = json.loads(line)
        rows.append({
            'frame_idx':         entry['frame_idx'],
            'timestamp_s':       entry['timestamp_s'],
            'n_persons':         len(entry['objects']),
            'n_unclassified':    len(entry['unclassified_motion_regions']),
            'flow_computed':     entry['flow_computed_this_frame'],
            'dynamic_px':        entry['dynamic_pixel_count'],
            'nearest_person_px': entry['nearest_person_px'],
        })

df = pd.DataFrame(rows)
print('First 10 frames of scene log:')
display(df.to_string(index=False))